In [ ]:
%cd /kaggle/working
import os

REPO_URL = "https://github.com/hassanimtiaz158/echolyx-MVP.git"
REPO_DIR = "/kaggle/working/echo"

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("Repo already present -> pulling latest changes")
    %cd {REPO_DIR}
    !git fetch origin main
    !git reset --hard origin/main
else:
    print("Cloning fresh")
    !git clone {REPO_URL} echo
    %cd {REPO_DIR}

!git log -1 --oneline

print("\nInstalling dependencies...")
!pip install -q -r requirements.txt

In [ ]:
import os
from collections import Counter
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
RAW = Path("/kaggle/working/echo/data/raw")
CHK = Path("/kaggle/working/echo/checkpoints")
RAW.mkdir(parents=True, exist_ok=True)
CHK.mkdir(parents=True, exist_ok=True)


def _link(dest: Path, target) -> bool:
    """Symlink dest -> target, replacing a stale symlink but never a real file/dir."""
    if target is None or not Path(target).exists():
        print(f"  !! {dest.name}: NOT FOUND")
        return False
    if dest.is_symlink():
        os.unlink(dest)
    elif dest.exists():
        print(f"  !! {dest} exists as a real file/dir -- leaving it")
        return False
    os.symlink(str(target), str(dest), target_is_directory=Path(target).is_dir())
    print(f"  {dest.name} -> {target}")
    return True


# Auto-discover data under /kaggle/input by CONTENT signature, not by dataset
# slug name -- Kaggle dataset names/bundling change over time and drift out of
# sync with hardcoded paths (that was the bug in the previous version of this
# cell). This works no matter how the datasets are attached/named.

# 1) MIMII archive roots: any directory containing attributes_00.csv.
#    DUE (fan) ships only section_00; DG (fan_dg) ships section_00/01/02 --
#    verified against both source zips, so the section count reliably tells
#    the two archives apart regardless of what Kaggle names the folders.
mimii_roots = sorted({p.parent for p in INPUT_ROOT.rglob("attributes_00.csv")})
due_root, dg_root = None, None
for root in mimii_roots:
    is_dg = any(root.rglob("*section_01*")) or any(root.rglob("*section_02*"))
    print(f"MIMII root found: {root}  -> {'DG (fan_dg)' if is_dg else 'DUE (fan)'}")
    if is_dg:
        dg_root = dg_root or root
    else:
        due_root = due_root or root

# 2) Real-world broken-fan holdout set: a directory literally named broken_fans.
broken_hits = [p for p in INPUT_ROOT.rglob("*") if p.is_dir() and p.name == "broken_fans"]
broken_root = broken_hits[0] if broken_hits else None

# 3) Freesound normal clips: the directory holding the most standalone .mp3
#    files. (Originally keyed off the sanity-clip filename "fan's frame
#    broken.mp3", but apostrophes routinely get mangled/dropped on
#    zip/Kaggle upload, so match by content instead of that one filename.)
mp3_counts = Counter(p.parent for p in INPUT_ROOT.rglob("*.mp3"))
mp3_counts.pop(broken_root, None)
freesound_root = max(mp3_counts, key=mp3_counts.get) if mp3_counts else None
if freesound_root:
    print(f"Freesound root found: {freesound_root}  ({mp3_counts[freesound_root]} mp3 files)")

# 4) PANNs backbone checkpoint.
ckpt_hits = list(INPUT_ROOT.rglob("Cnn14*.pth"))
ckpt_root = ckpt_hits[0] if ckpt_hits else None

ok = True
ok &= _link(RAW / "mimii" / "fan", due_root)
ok &= _link(RAW / "mimii" / "fan_dg", dg_root)
ok &= _link(RAW / "freesound", freesound_root)
ok &= _link(RAW / "broken_fans", broken_root)
ok &= _link(CHK / "Cnn14_mAP=0.431.pth", ckpt_root)
print("\nAll links OK:", ok)
if not ok:
    raise RuntimeError(
        "Missing data mount(s) above -- attach the dataset(s) with the "
        "missing content under Data > Add Input, then re-run this cell."
    )

%cd /kaggle/working/echo
!python -m src.data.collect --config configs/config.yaml
!python -m src.train --config configs/config.yaml
!python -m src.evaluate --config configs/config.yaml
!python -m src.anomaly --config configs/config.yaml

In [ ]:
%cd /kaggle/working/echo
import os

os.environ["GIT_TERMINAL_PROMPT"] = "0"
!git config user.email "kaggle@echolyx.ai"
!git config user.name "Kaggle Bot"
!git add -A -- checkpoints artifacts

status = !git status --porcelain
if status:
    print("Changed files:")
    !git status --short
    !git commit -m "chore: update training results from Kaggle run"

    # Token comes from a Kaggle Secret (Add-ons > Secrets, name it GITHUB_TOKEN
    # and enable it for this notebook) -- never hardcode a token in the cell.
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    remote = f"https://{token}@github.com/hassanimtiaz158/echolyx-MVP.git"
    !git remote set-url origin {remote}
    !git push origin main
    !git remote set-url origin https://github.com/hassanimtiaz158/echolyx-MVP.git
    print("Pushed to GitHub!")
else:
    print("No changes to push.")